<a href="https://colab.research.google.com/github/rlakshmi2k24/Learning_journey_1/blob/main/Model3_visit_risk_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install xgboost
!pip install catboost
!pip install imbalanced-learn
!pip install shap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.9 MB/s eta 0:00:00


In [4]:
import pandas as pd
import numpy as np

df_visit_risk = pd.read_csv("/content/sample_data/healthcare_analytical_dataset.csv")

print(df_visit_risk.shape)
df_visit_risk.head()

(25000, 26)


,visit_id,patient_id,visit_date,department,visit_type,length_of_stay_hours,risk_score,doctor_id,age,gender,...,approved_amount,claim_status,payment_days,billing_date,revenue_gap,senior_citizen_flag,long_stay_flag,payment_delay_flag,claim_rejected_flag,approval_ratio
0,1,756,2025-10-18,Cardiology,ER,3.48,Low,169,90,M,...,0.00,Rejected,16.0,2025-06-18,23577.37,1,0,0,1,0.0
1,2,4102,2025-04-06,Orthopedics,OPD,15.31,High,148,30,M,...,38178.81,Paid,18.0,2025-10-09,0.00,0,0,0,0,1.0
2,3,2964,2025-07-13,ICU,ER,34.36,Low,153,25,F,...,5038.97,Paid,NaN,2025-01-20,0.00,0,0,0,0,1.0
3,4,4496,2025-11-19,Cardiology,ER,37.89,High,119,75,M,...,22813.34,Paid,16.0,2025-12-24,0.00,1,0,0,0,1.0
4,5,1930,2025-03-29,General,ICU,16.78,Medium,118,80,M,...,27106.95,Paid,14.0,2025-09-23,0.00,1,0,0,0,1.0


In [9]:
#Visit Count
visit_frequency = (
    df_visit_risk.groupby("patient_id")
      .size()
      .reset_index(name="visit_count")
)

df_visit_risk = df_visit_risk.merge(
    visit_frequency,
    on="patient_id",
    how="left"
)
#Previous Average Stay
patient_avg_los = (
    df_visit_risk.groupby("patient_id")
      ["length_of_stay_hours"]
      .mean()
      .reset_index(name="patient_avg_los")
)

df_visit_risk = df_visit_risk.merge(
    patient_avg_los,
    on="patient_id",
    how="left"
)
#Previous High Risk Ratio
patient_high_risk_rate = (
    df_visit_risk.groupby("patient_id")
      ["risk_score"]
      .apply(
          lambda x:
          (x=="High").mean()
      )
      .reset_index(
          name="patient_high_risk_rate"
      )
)

df_visit_risk = df_visit_risk.merge(
    patient_high_risk_rate,
    on="patient_id",
    how="left"
)
#Department Historical Risk
dept_high_risk_rate = (
    df_visit_risk.groupby("department")
      ["risk_score"]
      .apply(
          lambda x:
          (x=="High").mean()
      )
      .reset_index(
          name="dept_high_risk_rate"
      )
)

df_visit_risk = df_visit_risk.merge(
    dept_high_risk_rate,
    on="department",
    how="left"
)
#Doctor Historical Risk
doctor_risk_rate = (
    df_visit_risk.groupby("doctor_id")
      ["risk_score"]
      .apply(
          lambda x:
          (x=="High").mean()
      )
      .reset_index(
          name="doctor_high_risk_rate"
      )
)

df_visit_risk = df_visit_risk.merge(
    doctor_risk_rate,
    on="doctor_id",
    how="left"
)
#Age Buckets
df_visit_risk["age_bucket"] = pd.cut(
    df_visit_risk["age"],
    bins=[0,18,40,60,120],
    labels=[
        "Child",
        "Adult",
        "Middle",
        "Senior"
    ]
)
#Chronic Patient Flag
df_visit_risk["high_risk_chronic"] = np.where(
    (df_visit_risk["chronic_flag"]==1)
    &
    (df_visit_risk["age"]>=60),
    1,
    0
)
#Age Risk Score
df_visit_risk["age_risk_score"] = np.select(
    [
        df_visit_risk["age"] < 18,
        df_visit_risk["age"] < 60,
        df_visit_risk["age"] >= 60
    ],
    [1,2,3]
)
#Chronic Disease Severity
df_visit_risk["chronic_age_interaction"] = (
    df_visit_risk["chronic_flag"]
    *
    df_visit_risk["age"]
)

#Department Risk Ranking
dept_rank = (
   df_visit_risk.groupby("department")
      ["risk_score"]
      .apply(lambda x:(x=="High").mean())
      .rank()
)

df_visit_risk["department_risk_rank"] = (
    df_visit_risk["department"]
      .map(dept_rank)
)
#Historical Visit Burden
df_visit_risk["patient_visit_burden"] = (
    np.log1p(df_visit_risk["visit_count"])
)
#Visit Type Risk
visit_risk = (
    df_visit_risk.groupby("visit_type")
      ["risk_score"]
      .apply(lambda x:(x=="High").mean())
)

df_visit_risk["visit_type_risk"] = (
    df_visit_risk["visit_type"]
      .map(visit_risk)
)

#Risk Model Dataset
target = "risk_score"

features = [

    "age",
    "gender",
    "city",
    "insurance_provider",
    "chronic_flag",
    "department",
    "visit_type",
    "doctor_id",
    "visit_count",
    "patient_avg_los",
    "patient_high_risk_rate",
    "dept_high_risk_rate",
    "doctor_high_risk_rate",
    "age_bucket",
    "high_risk_chronic",
    "age_risk_score",
    "chronic_age_interaction",
    "department_risk_rank",
    "patient_visit_burden",
    "visit_type_risk"
]

risk_df = (
    df_visit_risk[features + [target]]
      .dropna()
)

In [11]:
from sklearn.preprocessing import LabelEncoder

X = risk_df[features]

y = risk_df[target]

categorical_cols = (
    X.select_dtypes(
        include=[
            "object",
            "category"
        ]
    )
    .columns
)

encoders = {}

for col in categorical_cols:

    le = LabelEncoder()

    X[col] = le.fit_transform(
        X[col].astype(str)
    )

    encoders[col] = le

target_encoder = LabelEncoder()

y_encoded = target_encoder.fit_transform(y)

print(
    dict(
        zip(
            target_encoder.classes_,
            range(
                len(
                    target_encoder.classes_
                )
            )
        )
    )
)

{'High': 0, 'Low': 1, 'Medium': 2}


/tmp/ipykernel_1545/2741453598.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = le.fit_transform(
/tmp/ipykernel_1545/2741453598.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = le.fit_transform(
/tmp/ipykernel_1545/2741453598.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.

In [12]:
from sklearn.feature_selection import mutual_info_classif

mi_scores = mutual_info_classif(
    X,
    y_encoded
)

mi_df = pd.DataFrame({
    "Feature": X.columns,
    "Score": mi_scores
})

mi_df.sort_values(
    "Score",
    ascending=False
)

,Feature,Score
10,patient_high_risk_rate,0.120888
6,visit_type,0.008630
16,chronic_age_interaction,0.005749
2,city,0.004624
17,department_risk_rank,0.004588
19,visit_type_risk,0.004529
15,age_risk_score,0.004312
4,chronic_flag,0.004125
12,doctor_high_risk_rate,0.003607
7,doctor_id,0.003353


In [13]:
selected_features = (
    mi_df
      .head(12)
      ["Feature"]
      .tolist()
)

X = X[selected_features]

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y_encoded,
        test_size=0.20,
        stratify=y_encoded,
        random_state=42
    )
)

In [15]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(
    random_state=42
)

X_train_smote, y_train_smote = (
    smote.fit_resample(
        X_train,
        y_train
    )
)

In [16]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

rf_1 = RandomForestClassifier(
    random_state=42,
    class_weight="balanced"
)

rf_grid_1 = {

    "n_estimators":[
        200,
        300,
        500
    ],

    "max_depth":[
        5,
        10,
        15,
        None
    ],

    "min_samples_split":[
        2,
        5,
        10
    ]
}

rf_search_1 = RandomizedSearchCV(
    rf_1,
    rf_grid_1,
    scoring="f1_macro",
    cv=cv,
    n_iter=20,
    n_jobs=-1,
    random_state=42
)

rf_search_1.fit(
    X_train_smote,
    y_train_smote
)

best_rf_1 = rf_search_1.best_estimator_
print("Best Random Forest Parameters:")
print(rf_search_1.best_params_)

Best Random Forest Parameters:
{'n_estimators': 300, 'min_samples_split': 5, 'max_depth': None}


In [18]:
from xgboost import XGBClassifier

xgb_1 = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    num_class=3,
    random_state=42
)

xgb_grid_1 = {

    "n_estimators":[
        200,
        300,
        500
    ],

    "max_depth":[
        3,
        5,
        7
    ],

    "learning_rate":[
        0.01,
        0.05,
        0.1
    ],

    "subsample":[
        0.8,
        1.0
    ]
}

xgb_search_1 = RandomizedSearchCV(
    xgb_1,
    xgb_grid_1,
    scoring="f1_macro",
    cv=cv,
    n_iter=20,
    n_jobs=-1,
    random_state=42
)

xgb_search_1.fit(
    X_train_smote,
    y_train_smote
)

best_xgb_1 = xgb_search_1.best_estimator_
print("Best xgboost Parameters:")
print(xgb_search_1.best_params_)

Best xgboost Parameters:
{'subsample': 0.8, 'n_estimators': 500, 'max_depth': 7, 'learning_rate': 0.1}


In [19]:
from catboost import CatBoostClassifier

cat_1 = CatBoostClassifier(
    verbose=0,
    random_state=42
)

cat_grid_1 = {

    "depth":[
        4,
        6,
        8
    ],

    "learning_rate":[
        0.01,
        0.05,
        0.1
    ],

    "iterations":[
        200,
        500
    ]
}

cat_search_1 = RandomizedSearchCV(
    cat_1,
    cat_grid_1,
    scoring="f1_macro",
    cv=cv,
    n_iter=15,
    n_jobs=-1,
    random_state=42
)

cat_search_1.fit(
    X_train_smote,
    y_train_smote
)

best_cat_1 = cat_search_1.best_estimator_
print("Best Catboost Parameters:")
print(cat_search_1.best_params_)

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best Catboost Parameters:
{'learning_rate': 0.1, 'iterations': 500, 'depth': 8}


In [20]:
from sklearn.metrics import (
    accuracy_score,
    f1_score
)

models_1 = {

    "Random Forest": best_rf_1,

    "XGBoost": best_xgb_1,

    "CatBoost": best_cat_1
}

results_1 = []

for name, model_1 in models_1.items():

    pred_1 = model_1.predict(X_test)

    results_1.append({

        "Model": name,

        "Accuracy":
        accuracy_score(
            y_test,
            pred_1
        ),

        "F1 Macro":
        f1_score(
            y_test,
            pred_1,
            average="macro"
        )
    })

results_df_1 = pd.DataFrame(results_1)

results_df_1.sort_values(
    "F1 Macro",
    ascending=False
)

,Model,Accuracy,F1 Macro
2,CatBoost,0.4824,0.421391
0,Random Forest,0.4674,0.413722
1,XGBoost,0.4630,0.403093


In [22]:
df_visit_risk["risk_score"].value_counts()
mi_df.head(20)

,Feature,Score
0,age,0.003148
1,gender,0.003143
2,city,0.004624
3,insurance_provider,0.000000
4,chronic_flag,0.004125
5,department,0.000000
6,visit_type,0.008630
7,doctor_id,0.003353
8,visit_count,0.000000
9,patient_avg_los,0.001744


Analysis
Top Feature MI Score
0.20 - 0.60

best feature:

patient_high_risk_rate = 0.1208

Everything else is almost noise.
What This Means?
The model is basically learning from:
patient_high_risk_rate and almost nothing else.

This is why:
Random Forest = 0.41
XGBoost = 0.40
CatBoost = 0.42
The bottle neck is with the Feature Quality but not the Algorithm

In [23]:
# Verify Risk Score Distribution
print(risk_df["risk_score"].value_counts())

print(
    risk_df["risk_score"].value_counts(normalize=True)
)

risk_score
Low       12470
Medium     7496
High       5034
Name: count, dtype: int64
risk_score
Low       0.49880
Medium    0.29984
High      0.20136
Name: proportion, dtype: float64


Risk score disctribution shows

The target is not perfectly balanced, which is realistic for hospitals.

Most visits should indeed be Low Risk.

The dataset is moderately imbalanced:

Low : Medium : High
50 : 30 : 20

A naïve model predicting only:

Low

would already achieve:

~50% Accuracy

CatBoost accuracy is:

48.2%

which is actually worse than the majority-class baseline.

This strongly suggests that the model is struggling to find meaningful patterns.

In [24]:
#Check the Target Itself
risk_profile = (
    df_visit_risk.groupby("risk_score")
      .agg({
          "age":["mean","median"],
          "length_of_stay_hours":["mean","median"],
          "chronic_flag":"mean",
          "visit_count":"mean"
      })
)

risk_profile

age        length_of_stay_hours         chronic_flag  \
                 mean median                 mean  median         mean   
risk_score                                                               
High        44.781287   45.0            19.758576  18.215     0.494835   
Low         44.692141   45.0            19.151184  18.080     0.503448   
Medium      44.886339   45.0            20.078663  18.520     0.506403   

           visit_count  
                  mean  
risk_score              
High          5.933055  
Low           5.997835  
Medium        5.918090

| Risk   | Avg Age | Median Age | Avg LOS | Median LOS | Chronic % | Avg Visits |
| ------ | ------: | ---------: | ------: | ---------: | --------: | ---------: |
| High   |   44.78 |         45 |   19.76 |      18.21 |    49.48% |       5.93 |
| Low    |   44.69 |         45 |   19.15 |      18.08 |    50.34% |       5.99 |
| Medium |   44.89 |         45 |   20.08 |      18.52 |    50.64% |       5.92 |

The dataset has simlar values across 3 different classes

Age:
High   = 44.78
Low    = 44.69
Medium = 44.89

Difference < 0.2 years
Length of Stay:
High   = 19.76
Low    = 19.15
Medium = 20.08

Difference < 1 hour
Chronic Flag:
High   = 49.48%
Low    = 50.34%
Medium = 50.64%

Difference < 1.2%
Visit Count:
High   = 5.93
Low    = 5.99
Medium = 5.92

Almost identical
Why F1 = 0.42 is Actually Expected

A doctor trying to classify:

Patient A:
Age = 45
LOS = 19
Chronic = Yes

Patient B:
Age = 45
LOS = 20
Chronic = Yes

One is labeled:High

and another:Low

There is no learnable pattern.

Even humans cannot reliably distinguish them.

Therefore:

CatBoost ≈ 0.42
RandomForest ≈ 0.41
XGBoost ≈ 0.40



In [32]:
# copy risk_df to another data frame
risk_df_1 = risk_df.copy()

# Rebuild the Risk Score using Healthcare Business Logic.
def calculate_risk(row):

    score = 0

    if row["age"] >= 65:
        score += 2

    if row["chronic_flag"] == 1:
        score += 2

    if row["length_of_stay_hours"] > 24:
        score += 2

    if row["visit_count"] > 8:
        score += 1

    if score >= 5:
        return "High"

    elif score >= 3:
        return "Medium"

    else:
        return "Low"

risk_df_1["risk_score_v2"] = risk_df_1.apply(
    calculate_risk,
    axis=1
)
risk_df_1.head(20)

,age,gender,city,insurance_provider,chronic_flag,department,visit_type,doctor_id,visit_count,patient_avg_los,...,age_bucket,high_risk_chronic,age_risk_score,chronic_age_interaction,department_risk_rank,patient_visit_burden,visit_type_risk,length_of_stay_hours,risk_score,risk_score_v2
0,90,M,Bangalore,CareOne,1,Cardiology,ER,169,2,3.725000,...,Senior,1,3,90,1.0,1.098612,0.205917,3.48,Low,Medium
1,30,M,Hyderabad,SecureLife,1,Orthopedics,OPD,148,4,32.025000,...,Adult,0,2,30,3.0,1.609438,0.200931,15.31,High,Low
2,25,F,Chennai,HealthPlus,1,ICU,ER,153,4,20.542500,...,Adult,0,2,25,6.0,1.609438,0.205917,34.36,Low,Medium
3,75,M,Delhi,MediCareX,0,Cardiology,ER,119,7,28.165714,...,Senior,0,3,0,1.0,2.079442,0.205917,37.89,High,Medium
4,80,M,Bangalore,HealthPlus,1,General,ICU,118,5,22.988000,...,Senior,1,3,80,2.0,1.791759,0.197159,16.78,Medium,Medium
5,32,M,Bangalore,CareOne,0,General,OPD,157,3,21.033333,...,Adult,0,2,0,2.0,1.386294,0.200931,0.70,High,Low
6,10,F,Bangalore,CareOne,1,Cardiology,OPD,121,8,10.225000,...,Child,0,1,10,1.0,2.197225,0.200931,6.40,Low,Low
7,65,M,Mumbai,CareOne,0,ER,ICU,107,5,20.314000,...,Senior,0,3,0,5.0,1.791759,0.197159,26.86,High,Medium
8,51,M,Mumbai,HealthPlus,0,General,ER,134,7,24.697143,...,Middle,0,2,0,2.0,2.079442,0.205917,8.31,Medium,Low
9,49,F,Delhi,HealthPlus,1,ER,ER,107,9,18.795556,...,Middle,0,2,49,5.0,2.302585,0.205917,23.22,Low,Medium


In [65]:
#Risk Model Dataset
target = "risk_score_v2"

features = [

    "age",
    "gender",
    "city",
    "insurance_provider",
    "chronic_flag",
    "department",
    "visit_type",
    "doctor_id",
    "visit_count",
    "patient_avg_los",
    "patient_high_risk_rate",
    "dept_high_risk_rate",
    "doctor_high_risk_rate",
    "age_bucket",
    "high_risk_chronic",
    "age_risk_score",
    "chronic_age_interaction",
    "department_risk_rank",
    "patient_visit_burden",
    "visit_type_risk",



]

risk_df_1 = (
    risk_df_1[features + [target]]
      .dropna()
)

In [66]:
#preparing your data for Machine Learning by converting
#text/categorical values into numbers,because
#algorithms such as
#Random Forest,
#XGBoost,
#CatBoost,
#Logistic Regression,
#and Decision Trees cannot work directly with text values.

from sklearn.preprocessing import LabelEncoder

X = risk_df_1[features]

y = risk_df_1[target]

categorical_cols = (
    X.select_dtypes(
        include=[
            "object",
            "category"
        ]
    )
    .columns
)

encoders = {}

for col in categorical_cols:

    le = LabelEncoder()

    X[col] = le.fit_transform(
        X[col].astype(str)
    )

    encoders[col] = le

target_encoder = LabelEncoder()

y_encoded = target_encoder.fit_transform(y)

print(
    dict(
        zip(
            target_encoder.classes_,
            range(
                len(
                    target_encoder.classes_
                )
            )
        )
    )
)

{'High': 0, 'Low': 1, 'Medium': 2}


/tmp/ipykernel_1545/3253263845.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = le.fit_transform(
/tmp/ipykernel_1545/3253263845.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = le.fit_transform(
/tmp/ipykernel_1545/3253263845.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.

In [67]:
# performing Feature Selection using Mutual Information (MI)
# to determine which features are most useful for predicting the target variable.
from sklearn.feature_selection import mutual_info_classif

mi_scores = mutual_info_classif(
    X,
    y_encoded
)

mi_df = pd.DataFrame({
    "Feature": X.columns,
    "Score": mi_scores
})

mi_df.sort_values(
    "Score",
    ascending=False
)

,Feature,Score
9,patient_avg_los,0.243075
16,chronic_age_interaction,0.179493
4,chronic_flag,0.112405
0,age,0.075771
18,patient_visit_burden,0.074226
8,visit_count,0.072106
14,high_risk_chronic,0.070118
13,age_bucket,0.051324
15,age_risk_score,0.047369
10,patient_high_risk_rate,0.042610


In [68]:
selected_features = (
    mi_df
      .head(12)
      ["Feature"]
      .tolist()
)

X = X[selected_features]

In [69]:
from sklearn.model_selection import train_test_split

X_train_1, X_test_1, y_train_1, y_test_1 = (
    train_test_split(
        X,
        y_encoded,
        test_size=0.20,
        stratify=y_encoded,
        random_state=42
    )
)

In [70]:
from imblearn.over_sampling import SMOTE

smote_1 = SMOTE(
    random_state=42
)

X_train_smote_1, y_train_smote_1 = (
    smote_1.fit_resample(
        X_train_1,
        y_train_1
    )
)

In [71]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [78]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

rf_2 = RandomForestClassifier(
    random_state=42,
    class_weight="balanced"
)

rf_grid_2 = {

    "n_estimators":[
        200,
        300,
        500
    ],

    "max_depth":[
        5,
        10,
        15,
        None
    ],

    "min_samples_split":[
        2,
        5,
        10
    ]
}

rf_search_2 = RandomizedSearchCV(
    rf_2,
    rf_grid_2,
    scoring="f1_macro",
    cv=cv,
    n_iter=20,
    n_jobs=-1,
    random_state=42
)

rf_search_2.fit(
    X_train_smote,
    y_train_smote
)

best_rf_2 = rf_search_2.best_estimator_
print("Best Random Forest Parameters:")
print(rf_search_2.best_params_)

Best Random Forest Parameters:
{'n_estimators': 300, 'min_samples_split': 5, 'max_depth': None}


In [77]:
from xgboost import XGBClassifier

xgb_2 = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    num_class=3,
    random_state=42
)

xgb_grid_2 = {

    "n_estimators":[
        200,
        300,
        500
    ],

    "max_depth":[
        3,
        5,
        7
    ],

    "learning_rate":[
        0.01,
        0.05,
        0.1
    ],

    "subsample":[
        0.8,
        1.0
    ]
}

xgb_search_2 = RandomizedSearchCV(
    xgb_2,
    xgb_grid_2,
    scoring="f1_macro",
    cv=cv,
    n_iter=20,
    n_jobs=-1,
    random_state=42
)

xgb_search_2.fit(
    X_train_smote_1,
    y_train_smote_1
)

best_xgb_2 = xgb_search_2.best_estimator_
print("Best xgboost Parameters:")
print(xgb_search_2.best_params_)

Best xgboost Parameters:
{'subsample': 0.8, 'n_estimators': 500, 'max_depth': 7, 'learning_rate': 0.1}


In [72]:
from catboost import CatBoostClassifier

cat_2 = CatBoostClassifier(
    verbose=0,
    random_state=42
)

cat_grid_2 = {

    "depth":[
        4,
        6,
        8
    ],

    "learning_rate":[
        0.01,
        0.05,
        0.1
    ],

    "iterations":[
        200,
        500
    ]
}

cat_search_2 = RandomizedSearchCV(
    cat_2,
    cat_grid_2,
    scoring="f1_macro",
    cv=cv,
    n_iter=15,
    n_jobs=-1,
    random_state=42
)

cat_search_2.fit(
    X_train_smote_1,
    y_train_smote_1
)

best_cat_2 = cat_search_2.best_estimator_
print("Best Catboost Parameters:")
print(cat_search_2.best_params_)

Best Catboost Parameters:
{'learning_rate': 0.1, 'iterations': 500, 'depth': 8}


In [73]:
#Print F1 score for Cat boost
from sklearn.metrics import (
    accuracy_score,
    f1_score
)
#print f1_score
print(f1_score(y_test_1, best_cat_2.predict(X_test_1), average='macro'))
#print confusion matrix
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test_1, best_cat_2.predict(X_test_1)))
#print accuracy, recall percentage
print(f"Accuracy: {accuracy_score(y_test_1, best_cat_2.predict(X_test_1))}")



0.6177280241478599
[[ 113    0  128]
 [   0 3142  376]
 [ 151  470  620]]
Accuracy: 0.775


In [79]:
from sklearn.metrics import (
    accuracy_score,
    f1_score
)

models_2 = {
    "Random Forest": best_rf_2,

    "XGBoost": best_xgb_2,

    "CatBoost": best_cat_2
}

results_2 = []

for name, model_2 in models_2.items():

    pred_2 = model_2.predict(X_test_1)

    results_2.append({

        "Model": name,

        "Accuracy":
        accuracy_score(
            y_test_1,
            pred_2
        ),

        "F1 Macro":
        f1_score(
            y_test_1,
            pred_2,
            average="macro"
        )
    })

results_df_2 = pd.DataFrame(results_2)

results_df_2.sort_values(
    "F1 Macro",
    ascending=False
)

,Model,Accuracy,F1 Macro
2,CatBoost,0.7750,0.617728
1,XGBoost,0.7726,0.606212
0,Random Forest,0.4402,0.304394


In [80]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    pred_1
))

              precision    recall  f1-score   support

           0       0.41      0.42      0.42      1007
           1       0.54      0.68      0.60      2494
           2       0.35      0.19      0.24      1499

    accuracy                           0.48      5000
   macro avg       0.43      0.43      0.42      5000
weighted avg       0.46      0.48      0.46      5000



In [81]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test_1,
    pred_2
))

              precision    recall  f1-score   support

           0       0.43      0.47      0.45       241
           1       0.87      0.89      0.88      3518
           2       0.55      0.50      0.52      1241

    accuracy                           0.78      5000
   macro avg       0.62      0.62      0.62      5000
weighted avg       0.77      0.78      0.77      5000

